# Notebook 6 — H1 Stability: Per-Method Bootstrap and Pre-KNN Core Jaccard

**Project:** Data-Driven Cognitive Phenotyping in Acquired Brain Injury  
**Author:** Zoltan Kunos | Universitat de Barcelona  

Adds two stability analyses for H1:

- **#1 Per-cluster core Jaccard** (Hennig 2007 matching on pre-KNN HDBSCAN labels): isolates cluster reproducibility from boundary-driven label flips that the original full-coverage Jaccard conflated.
- **#3 Per-method bootstrap thresholds**: each imputation method is bootstrap-resampled with HDBSCAN re-fitted on the cached UMAP embedding; a method passes H1 if median silhouette $> 0.40$ and median pre-KNN noise $< 0.30$.

Outputs: `results/h1_improved.pkl`, `H1_Per_Method_Bootstrap.csv`, `H1_Core_Jaccard.csv`.

In [1]:
"""H1 improvements: per-method bootstrap stability and pre-KNN core Jaccard.

Two analyses:

#1 Core Jaccard (per cluster, Hennig 2007 matching):
    - Original cluster definition uses HDBSCAN labels BEFORE KNN noise
      reassignment (i.e., -1 for noise).
    - For each bootstrap resample, re-fit HDBSCAN on the resampled UMAP
      embedding, find the bootstrap cluster with highest Jaccard overlap
      against each original cluster, average per cluster across bootstrap
      iterations. Reports per-cluster Jaccard for each method.

#3 Per-method bootstrap thresholds:
    - For each method, B bootstrap resamples; on each, fit HDBSCAN, compute
      silhouette (subsampled) on non-noise points, and pre-KNN noise fraction.
    - A method "passes" H1 if median silhouette > 0.40 AND median noise < 0.30.
    - Reports the count of passing methods (target: >=8/10).

Outputs
    results/h1_improved.pkl
    results/H1_Per_Method_Bootstrap.csv
    results/H1_Core_Jaccard.csv
"""

from __future__ import annotations
import pickle, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import hdbscan
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"

RS = 42
B = 50           # bootstrap iterations per method
SIL_SUB = 5000   # subsample size for silhouette
ALPHA = 0.05

H1_SIL_THRESHOLD = 0.40
H1_NOISE_THRESHOLD = 0.30


def log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


# ---------------------------------------------------------------------------
# Load shared infrastructure
# ---------------------------------------------------------------------------
log("Loading shared infrastructure...")
with open(RESULTS / "shared_infrastructure.pkl", "rb") as f:
    infra = pickle.load(f)

METHODS = infra["METHODS"]
umap_embeddings = infra["umap_embeddings"]
hdbscan_params = infra["BEST_HDBSCAN_PARAMS"]
log(f"Best HDBSCAN params: {hdbscan_params}")
log(f"Methods: {METHODS}")


def fit_hdbscan(embedding: np.ndarray) -> np.ndarray:
    """Fit HDBSCAN with the project's best params; return labels with -1 = noise."""
    clu = hdbscan.HDBSCAN(
        min_cluster_size=hdbscan_params.get("min_cluster_size", 500),
        min_samples=hdbscan_params.get("min_samples", 50),
        cluster_selection_method=hdbscan_params.get("cluster_selection_method", "eom"),
        cluster_selection_epsilon=hdbscan_params.get("cluster_selection_epsilon", 0.0),
        prediction_data=False,
    )
    return clu.fit_predict(embedding)


def per_cluster_jaccard(orig_labels: np.ndarray, boot_labels: np.ndarray,
                        boot_indices: np.ndarray) -> dict:
    """Hennig (2007) per-cluster Jaccard.

    Both label arrays are pre-KNN HDBSCAN labels (with -1 for noise).
    For each non-noise cluster in `orig_labels`, find the bootstrap cluster
    on `boot_indices` with maximum Jaccard overlap and return that score.
    """
    # Restrict to indices that appear in the bootstrap (boot_indices contains
    # original-row indices, possibly duplicated). For a clean cluster-vs-cluster
    # Jaccard we deduplicate.
    uniq_idx = np.unique(boot_indices)
    boot_label_at_orig = np.full(orig_labels.shape, -2, dtype=int)
    # Map: for each unique original index that appears in the bootstrap,
    # take the label HDBSCAN assigned to it on the bootstrap-indexed embedding.
    # Because of duplicates, we take the modal label per unique index.
    for i in uniq_idx:
        mask = boot_indices == i
        labs = boot_labels[mask]
        if len(labs) == 0:
            continue
        # Modal label (ties broken by first occurrence)
        vals, counts = np.unique(labs, return_counts=True)
        boot_label_at_orig[i] = vals[np.argmax(counts)]

    # Now boot_label_at_orig has the bootstrap-derived label at every original
    # row that participated in the bootstrap; -2 for rows that didn't.
    out = {}
    orig_clusters = sorted(c for c in np.unique(orig_labels) if c != -1)
    for k in orig_clusters:
        orig_mask = (orig_labels == k)
        # Restrict comparison to rows that participated in the bootstrap
        considered = (boot_label_at_orig != -2)
        common_orig_mask = orig_mask & considered
        if common_orig_mask.sum() == 0:
            out[int(k)] = float("nan")
            continue
        # Find the bootstrap cluster that maximises Jaccard with orig cluster k
        best_j = 0.0
        for kb in np.unique(boot_label_at_orig[considered]):
            if kb == -1:  # bootstrap noise can't match an original cluster
                continue
            boot_mask = (boot_label_at_orig == kb) & considered
            inter = (orig_mask & boot_mask).sum()
            union = (orig_mask | boot_mask).sum()
            if union == 0:
                continue
            j = inter / union
            if j > best_j:
                best_j = j
        out[int(k)] = float(best_j)
    return out


# ---------------------------------------------------------------------------
# Loop over methods
# ---------------------------------------------------------------------------
results = {}
rng = np.random.default_rng(RS)

for method in METHODS:
    log(f"=== {method} ===")
    emb = np.asarray(umap_embeddings[method])
    n = len(emb)

    # Original pre-KNN labels (re-fit on full data, deterministic given params)
    t0 = time.time()
    orig_labels = fit_hdbscan(emb)
    n_clusters_orig = len(set(orig_labels)) - (1 if -1 in orig_labels else 0)
    noise_frac_orig = float((orig_labels == -1).mean())
    if n_clusters_orig >= 2:
        sil_orig = float(silhouette_score(
            emb[orig_labels != -1], orig_labels[orig_labels != -1],
            sample_size=min(SIL_SUB, (orig_labels != -1).sum()),
            random_state=RS,
        ))
    else:
        sil_orig = float("nan")
    log(f"  Full-data: {n_clusters_orig} clusters, sil={sil_orig:.4f}, "
        f"noise_pre_knn={noise_frac_orig:.3%}  ({time.time()-t0:.1f}s)")

    # Bootstrap loop
    sil_boot = []
    noise_boot = []
    cluster_count_boot = []
    cluster_jaccards = {int(c): [] for c in np.unique(orig_labels) if c != -1}

    t0 = time.time()
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        emb_b = emb[idx]
        labels_b = fit_hdbscan(emb_b)

        nf = float((labels_b == -1).mean())
        noise_boot.append(nf)
        n_clu_b = len(set(labels_b)) - (1 if -1 in labels_b else 0)
        cluster_count_boot.append(n_clu_b)

        # Silhouette on non-noise points
        nn = labels_b != -1
        if nn.sum() >= 2 and len(set(labels_b[nn])) >= 2:
            sil_b = silhouette_score(
                emb_b[nn], labels_b[nn],
                sample_size=min(SIL_SUB, int(nn.sum())),
                random_state=RS + b,
            )
        else:
            sil_b = float("nan")
        sil_boot.append(float(sil_b))

        # Per-cluster Jaccard
        per_cluster = per_cluster_jaccard(orig_labels, labels_b, idx)
        for k, j in per_cluster.items():
            cluster_jaccards.setdefault(k, []).append(j)

        if (b + 1) % 10 == 0:
            log(f"  boot {b+1}/{B}: sil={np.nanmedian(sil_boot):.3f} "
                f"noise={np.nanmedian(noise_boot):.3f} "
                f"({time.time()-t0:.1f}s)")

    sil_arr = np.asarray(sil_boot)
    noise_arr = np.asarray(noise_boot)
    sil_med = float(np.nanmedian(sil_arr))
    sil_mean = float(np.nanmean(sil_arr))
    noise_med = float(np.nanmedian(noise_arr))
    noise_mean = float(np.nanmean(noise_arr))
    passes = (sil_med > H1_SIL_THRESHOLD) and (noise_med < H1_NOISE_THRESHOLD)

    cluster_jaccard_means = {k: float(np.nanmean(v)) for k, v in cluster_jaccards.items()}
    overall_core_jaccard = float(np.nanmean(list(cluster_jaccard_means.values())))

    results[method] = dict(
        n_clusters_orig=int(n_clusters_orig),
        sil_full=sil_orig,
        noise_full=noise_frac_orig,
        sil_boot_median=sil_med,
        sil_boot_mean=sil_mean,
        noise_boot_median=noise_med,
        noise_boot_mean=noise_mean,
        passes_thresholds=bool(passes),
        per_cluster_jaccard=cluster_jaccard_means,
        overall_core_jaccard=overall_core_jaccard,
        cluster_count_boot=cluster_count_boot,
        sil_boot=sil_arr.tolist(),
        noise_boot=noise_arr.tolist(),
    )

    log(f"  RESULT: median sil={sil_med:.3f}, median noise={noise_med:.3%}, "
        f"core J={overall_core_jaccard:.3f}, passes={passes}")

# ---------------------------------------------------------------------------
# Aggregate and save
# ---------------------------------------------------------------------------
n_passing = sum(1 for r in results.values() if r["passes_thresholds"])
log(f"H1 per-method bootstrap: {n_passing}/{len(METHODS)} methods clear thresholds")

# Build CSVs
df_per_method = pd.DataFrame([
    dict(
        method=m,
        n_clusters_orig=r["n_clusters_orig"],
        sil_full=r["sil_full"],
        noise_full=r["noise_full"],
        sil_boot_median=r["sil_boot_median"],
        sil_boot_mean=r["sil_boot_mean"],
        noise_boot_median=r["noise_boot_median"],
        noise_boot_mean=r["noise_boot_mean"],
        passes_thresholds=r["passes_thresholds"],
        overall_core_jaccard=r["overall_core_jaccard"],
    )
    for m, r in results.items()
])
df_per_method.to_csv(RESULTS / "H1_Per_Method_Bootstrap.csv", index=False)

jaccard_rows = []
for m, r in results.items():
    for k, j in r["per_cluster_jaccard"].items():
        jaccard_rows.append(dict(method=m, cluster=k, core_jaccard=j))
df_jaccard = pd.DataFrame(jaccard_rows)
df_jaccard.to_csv(RESULTS / "H1_Core_Jaccard.csv", index=False)

with open(RESULTS / "h1_improved.pkl", "wb") as f:
    pickle.dump(dict(
        B=B,
        sil_threshold=H1_SIL_THRESHOLD,
        noise_threshold=H1_NOISE_THRESHOLD,
        n_passing=n_passing,
        n_methods=len(METHODS),
        per_method=results,
    ), f)

log(f"Saved results/h1_improved.pkl, H1_Per_Method_Bootstrap.csv, H1_Core_Jaccard.csv")
log("DONE.")


[00:24:55] Loading shared infrastructure...


[00:24:55] Best HDBSCAN params: {'min_cluster_size': 500, 'min_samples': 10, 'cluster_selection_method': 'eom', 'cluster_selection_epsilon': 0.0}


[00:24:55] Methods: ['Mean', 'KNN', 'MICE', 'MissForest', 'PMM', 'EM', 'SoftImpute', 'NMF', 'DAE', 'VAE']


[00:24:55] === Mean ===


[00:24:56]   Full-data: 7 clusters, sil=0.4818, noise_pre_knn=6.002%  (1.6s)


...

[00:29:19]   boot 50/50: sil=0.419 noise=0.144 (26.2s)


[00:29:19]   RESULT: median sil=0.419, median noise=14.365%, core J=0.537, passes=True


[00:29:19] H1 per-method bootstrap: 8/10 methods clear thresholds


[00:29:19] Saved results/h1_improved.pkl, H1_Per_Method_Bootstrap.csv, H1_Core_Jaccard.csv


[00:29:19] DONE.
